**ETL**

In [3]:
import os
from pathlib import Path

import pandas as pd

In [4]:
import os
import pandas as pd

**Extraindo e juntando os dados**

In [5]:
# Caminho dos arquivos com os dados
arquivos_path = Path("..") / "Datas"
arquivos_name_list = sorted([arquivo.name for arquivo in arquivos_path.glob("*.csv")])

# Carrega os arquivos como DataFrames do pandas
# Mantemos tudo como texto na leitura para preservar CNPJs e padronizar a limpeza depois.
tdf_dict = {}
for arquivo_name in arquivos_name_list:
    try:
        path = arquivos_path / arquivo_name
        tdf = pd.read_csv(path, sep=";", encoding="cp1252", dtype=str)
        tdf["arquivo_origem"] = arquivo_name
        tdf_dict[arquivo_name] = tdf
    except Exception as e:
        print(f"Erro ao ler o arquivo '{arquivo_name}': {e}")

In [6]:
# Junta os dados em um unico DataFrame
df = pd.concat(tdf_dict.values(), ignore_index=True)

# Padroniza nomes de colunas e remove espacos extras

# Remove espacos extras de todas as colunas de texto
# e converte campos de data e valor para tipos adequados.
df.columns = df.columns.str.strip()
text_columns = [
    coluna for coluna in df.columns
    if pd.api.types.is_string_dtype(df[coluna]) or df[coluna].dtype == object
]
for coluna in text_columns:
    df[coluna] = df[coluna].str.strip()

df["DatGeracaoConjuntoDados"] = pd.to_datetime(df["DatGeracaoConjuntoDados"], errors="coerce")
df["DatCompetencia"] = pd.to_datetime(df["DatCompetencia"], errors="coerce")
df["VlrMercado"] = pd.to_numeric(
    df["VlrMercado"].str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
    errors="coerce",
)

# CNPJ deve ser tratado como identificador textual
for coluna in ["NumCNPJAgenteDistribuidora", "NumCNPJAgenteAcessante"]:
    df[coluna] = (
        df[coluna]
        .str.replace(r"\D", "", regex=True)
        .replace({"": pd.NA})
    )

In [7]:
print("Contagem de valores unicos") #inclui os valores nulos
print(df.nunique(dropna=False))

Contagem de valores unicos
DatGeracaoConjuntoDados             1
NumCNPJAgenteDistribuidora        103
SigAgenteDistribuidora            103
NomAgenteDistribuidora            103
NomTipoMercado                     13
DscModalidadeTarifaria              7
DscSubGrupoTarifario               11
DscClasseConsumoMercado            10
DscSubClasseConsumidor             18
DscDetalheConsumidor               12
IdeAgenteAcessante                  1
NumCNPJAgenteAcessante            103
NomAgenteAcessante               1643
DscPostoTarifario                   5
DscOpcaoEnergia                     5
DscDetalheMercado                  25
DatCompetencia                     28
VlrMercado                    1119081
arquivo_origem                      3
dtype: int64


**Analizando e transformando dados**

Foram utilizados os dados de 2024 a 2026. O arquivo de 2026 ainda pode estar em atualização, então vale conferir o volume antes de usar comparações finais.

1. Informações gerais dos dados

In [8]:
df.head(5)

,DatGeracaoConjuntoDados,NumCNPJAgenteDistribuidora,SigAgenteDistribuidora,NomAgenteDistribuidora,NomTipoMercado,DscModalidadeTarifaria,DscSubGrupoTarifario,DscClasseConsumoMercado,DscSubClasseConsumidor,DscDetalheConsumidor,IdeAgenteAcessante,NumCNPJAgenteAcessante,NomAgenteAcessante,DscPostoTarifario,DscOpcaoEnergia,DscDetalheMercado,DatCompetencia,VlrMercado,arquivo_origem
0,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Sistema Isolado - Regular,Convencional,B1,Residencial,Residencial baixa renda – faixa 04,Não se aplica,NaN,37121669900,Não se aplica,Não se aplica,CATIVO,Energia TE (kWh),2024-01-01,583151.00,samp-2024.csv
1,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Regular,Convencional,B1,Residencial,Residencial baixa renda – faixa 01,Não se aplica,NaN,37121669900,Não se aplica,Não se aplica,CATIVO,PIS/PASEP (R$),2024-01-01,5431.29,samp-2024.csv
2,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Regular,Azul,A4,Serviço público,"Água, esgoto e saneamento",Não se aplica,NaN,37121669900,Não se aplica,Ponta,CATIVO,Receita Demanda (R$),2024-01-01,28137.26,samp-2024.csv
3,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Sistema de Compensação GD II,Branca,B1,Residencial,Residencial,Não se aplica,NaN,37121669900,Não se aplica,Fora ponta,CATIVO,Receita energia compensada (R$),2024-01-01,235.94,samp-2024.csv
4,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Regular,Convencional,B1,Residencial,Residencial baixa renda – faixa 04,Não se aplica,NaN,37121669900,Não se aplica,Não se aplica,CATIVO,Receita Energia (R$),2024-01-01,2061470.44,samp-2024.csv


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3029329 entries, 0 to 3029328
Data columns (total 19 columns):
 #   Column                      Dtype         
---  ------                      -----         
 0   DatGeracaoConjuntoDados     datetime64[us]
 1   NumCNPJAgenteDistribuidora  str           
 2   SigAgenteDistribuidora      str           
 3   NomAgenteDistribuidora      str           
 4   NomTipoMercado              str           
 5   DscModalidadeTarifaria      str           
 6   DscSubGrupoTarifario        str           
 7   DscClasseConsumoMercado     str           
 8   DscSubClasseConsumidor      str           
 9   DscDetalheConsumidor        str           
 10  IdeAgenteAcessante          str           
 11  NumCNPJAgenteAcessante      str           
 12  NomAgenteAcessante          str           
 13  DscPostoTarifario           str           
 14  DscOpcaoEnergia             str           
 15  DscDetalheMercado           str           
 16  DatCompetencia              d

Inicialmente é possivel observar que o conjunto de dados tem 3.780.041 linhas e 18 colunas; com 15 colunas sendo do tipo string, 2 sendo float64 e 1 sendo int64.

Segue uma descrição sobre cada coluna e o que ela representa:

01. DatGeracaoConjuntoDados:
02. NumCNPJAgenteDistribuidora: CNPJ da agencia distribuidora de energia.
03. SigAgenteDistribuidora: Sigla utilizada para idantificar a agencia distribuidora de energia.
04. NomAgenteDistribuidora: Nome completo da agencia distribuidora de energia.
05. NomTipoMercado: Identifica qual o tipo de mercado de energia elétrica do registro.
06. DscModalidadeTarifaria: Indica o tipo de cobrança aplicada ao fornecimento de energia elétrica daquele registro.
07. DscSubGrupoTarifario: Indica o subgrupo tarifário ao qual a modalidade tarifaria faz parte.
08. DscClasseConsumoMercado: 
09. DscSubClasseConsumidor:
10. DscDetalheConsumidor:
11. IdeAgenteAcessante:
12. NumCNPJAgenteAcessante:
13. NomAgenteAcessante:
14. DscPostoTarifario: indica o período do dia ao qual o consumo ou a demanda de energia está associado.
15. DscOpcaoEnergia:
16. DscDetalheMercado: Indica o que o registro está medindo no VlrMercado.
17. DatCompetencia:
18. VlrMercado:

In [10]:
postos_tarifarios = df["DscPostoTarifario"].dropna().unique()
print(postos_tarifarios)

<StringArray>
['Não se aplica', 'Ponta', 'Fora ponta', 'Intermediário', 'Fora Ponta']
Length: 5, dtype: str


2. Tipos dos dados

Ao observar os tipos é possivel notar que á 2 colunas com CNPJ (NumCNPJAgenteDistribuidora e NumCNPJAgenteAcessante) estão como int64 e float64, o que é ruim já que ele deve ser interpretado como um valor identificador e não como um numero.

In [11]:
df[["NumCNPJAgenteDistribuidora", "NumCNPJAgenteAcessante"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 3029329 entries, 0 to 3029328
Data columns (total 2 columns):
 #   Column                      Dtype
---  ------                      -----
 0   NumCNPJAgenteDistribuidora  str  
 1   NumCNPJAgenteAcessante      str  
dtypes: str(2)
memory usage: 46.2 MB


**3. Camada Silver - limpeza e padronização**

Nesta etapa tratamos valores nulos, removemos duplicidades, padronizamos textos e criamos colunas auxiliares para análise.

In [12]:
#Verificando valores nulos por coluna:

print("Valores ausentes por coluna:")
print(df.isnull().sum())

Valores ausentes por coluna:
DatGeracaoConjuntoDados             0
NumCNPJAgenteDistribuidora          0
SigAgenteDistribuidora              0
NomAgenteDistribuidora              0
NomTipoMercado                      0
DscModalidadeTarifaria              0
DscSubGrupoTarifario                0
DscClasseConsumoMercado             0
DscSubClasseConsumidor              0
DscDetalheConsumidor                0
IdeAgenteAcessante            3029329
NumCNPJAgenteAcessante         173586
NomAgenteAcessante                  0
DscPostoTarifario                   0
DscOpcaoEnergia                     0
DscDetalheMercado                   0
DatCompetencia                      0
VlrMercado                          0
arquivo_origem                      0
dtype: int64


In [13]:
silver_df = df.copy()

# Registra metricas antes de alterar a base.
linhas_duplicadas_exatas = int(silver_df.duplicated().sum())
silver_df_sem_dup = silver_df.drop_duplicates().reset_index(drop=True)
linhas_sem_competencia_ou_valor = int(silver_df_sem_dup[["DatCompetencia", "VlrMercado"]].isna().any(axis=1).sum())

# Remove duplicidades exatas.
silver_df = silver_df_sem_dup.copy()

# Normaliza textos para evitar inconsistencias de capitalização/espacos.
textual_columns = [
    coluna for coluna in silver_df.columns
    if pd.api.types.is_string_dtype(silver_df[coluna])
]
for coluna in textual_columns:
    silver_df[coluna] = (
        silver_df[coluna]
         .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.upper() #evita registros duplicados por diferenças de letras
    )

# Mapeamento para unificar strings equivalentes que gerariam IDs duplicados falsos
padronizacao_termos = {
    "NÃO SE APLICA": "NÃO SE APLICA",
    "NAO SE APLICA": "NÃO SE APLICA",
    "REGULAR": "REGULAR",
    "SISTEMA DE COMPENSAÇÃO GD I": "SISTEMA DE COMPENSAÇÃO GD I"
}

# Aplicando a padronização nas colunas críticas textuais para garantir unicidade real
colunas_para_ajuste = ["NomTipoMercado", "DscClasseConsumoMercado", "DscModalidadeTarifaria"]
for col in colunas_para_ajuste:
    if col in silver_df.columns:
        silver_df[col] = silver_df[col].replace(padronizacao_termos)
    






# Remove registros sem valor de mercado e sem data de competencia, pois nao sao uteis para agregacao.
silver_df = silver_df.dropna(subset=["DatCompetencia", "VlrMercado"])

# Colunas auxiliares para analise temporal.
silver_df["competencia_ano"] = silver_df["DatCompetencia"].dt.year
silver_df["competencia_mes"] = silver_df["DatCompetencia"].dt.to_period("M").astype(str)
silver_df["competencia_trimestre"] = silver_df["DatCompetencia"].dt.to_period("Q").astype(str)

# Padroniza colunas de identificacao como texto sem simbolos.
for coluna in ["NumCNPJAgenteDistribuidora", "NumCNPJAgenteAcessante"]:
    silver_df[coluna] = silver_df[coluna].astype("string")

# Preenche nulos dos atributos de acessante para evitar problemas em filtros de BI.
silver_df["IdeAgenteAcessante"] = silver_df["IdeAgenteAcessante"].fillna("Nao Informado")
silver_df["NomAgenteAcessante"] = silver_df["NomAgenteAcessante"].fillna("Nao Informado")
silver_df["NumCNPJAgenteAcessante"] = silver_df["NumCNPJAgenteAcessante"].fillna("00000000000000")

silver_df.head()

,DatGeracaoConjuntoDados,NumCNPJAgenteDistribuidora,SigAgenteDistribuidora,NomAgenteDistribuidora,NomTipoMercado,DscModalidadeTarifaria,DscSubGrupoTarifario,DscClasseConsumoMercado,DscSubClasseConsumidor,DscDetalheConsumidor,...,NomAgenteAcessante,DscPostoTarifario,DscOpcaoEnergia,DscDetalheMercado,DatCompetencia,VlrMercado,arquivo_origem,competencia_ano,competencia_mes,competencia_trimestre
0,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,SISTEMA ISOLADO - REGULAR,CONVENCIONAL,B1,RESIDENCIAL,RESIDENCIAL BAIXA RENDA – FAIXA 04,NÃO SE APLICA,...,NÃO SE APLICA,NÃO SE APLICA,CATIVO,ENERGIA TE (KWH),2024-01-01,583151.00,SAMP-2024.CSV,2024,2024-01,2024Q1
1,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,REGULAR,CONVENCIONAL,B1,RESIDENCIAL,RESIDENCIAL BAIXA RENDA – FAIXA 01,NÃO SE APLICA,...,NÃO SE APLICA,NÃO SE APLICA,CATIVO,PIS/PASEP (R$),2024-01-01,5431.29,SAMP-2024.CSV,2024,2024-01,2024Q1
2,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,REGULAR,AZUL,A4,SERVIÇO PÚBLICO,"ÁGUA, ESGOTO E SANEAMENTO",NÃO SE APLICA,...,NÃO SE APLICA,PONTA,CATIVO,RECEITA DEMANDA (R$),2024-01-01,28137.26,SAMP-2024.CSV,2024,2024-01,2024Q1
3,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,SISTEMA DE COMPENSAÇÃO GD II,BRANCA,B1,RESIDENCIAL,RESIDENCIAL,NÃO SE APLICA,...,NÃO SE APLICA,FORA PONTA,CATIVO,RECEITA ENERGIA COMPENSADA (R$),2024-01-01,235.94,SAMP-2024.CSV,2024,2024-01,2024Q1
4,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,REGULAR,CONVENCIONAL,B1,RESIDENCIAL,RESIDENCIAL BAIXA RENDA – FAIXA 04,NÃO SE APLICA,...,NÃO SE APLICA,NÃO SE APLICA,CATIVO,RECEITA ENERGIA (R$),2024-01-01,2061470.44,SAMP-2024.CSV,2024,2024-01,2024Q1


In [14]:
# Indicadores de qualidade da camada Silver.
resumo_qualidade = pd.DataFrame({
    "metric": [
        "linhas_bronze",
        "linhas_silver",
        "duplicatas_removidas",
        "linhas_sem_competencia_ou_valor",
        "linhas_descartadas_por_nulos",
        "datas_competencia_invalidas",
        "valores_nulos_vlrmercado",
    ],
    "value": [
        len(df),
        len(silver_df),
        linhas_duplicadas_exatas,
        linhas_sem_competencia_ou_valor,
        linhas_sem_competencia_ou_valor,
        int(df["DatCompetencia"].isna().sum()),
        int(df["VlrMercado"].isna().sum()),
    ],
})

resumo_qualidade

,metric,value
0,linhas_bronze,3029329
1,linhas_silver,3029186
2,duplicatas_removidas,143
3,linhas_sem_competencia_ou_valor,0
4,linhas_descartadas_por_nulos,0
5,datas_competencia_invalidas,0
6,valores_nulos_vlrmercado,0


**4. Camada Gold - esquema estrela**

Nesta etapa transformamos a camada Silver em dimensões e fato para consultas analíticas mais consistentes.

In [15]:
processed_root = arquivos_path / "processed"
bronze_dir = processed_root / "bronze"
silver_dir = processed_root / "silver"
gold_dir = processed_root / "gold"

bronze_dir.mkdir(parents=True, exist_ok=True)
silver_dir.mkdir(parents=True, exist_ok=True)
gold_dir.mkdir(parents=True, exist_ok=True)

bronze_path = bronze_dir / "bronze_dataset.csv"
silver_path = silver_dir / "silver_dataset.csv"
dim_tempo_path = gold_dir / "dim_tempo.csv"
dim_distribuidora_path = gold_dir / "dim_distribuidora.csv"
dim_acessante_path = gold_dir / "dim_acessante.csv"
dim_mercado_path = gold_dir / "dim_mercado.csv"
fato_energia_path = gold_dir / "fato_energia.csv"

# Dimensão tempo
dim_tempo = (
    silver_df[["DatCompetencia"]]
    .drop_duplicates()
    .dropna()
    .sort_values("DatCompetencia")
    .reset_index(drop=True)
    .assign(
        tempo_id=lambda tabela: tabela.index + 1,
        ano=lambda tabela: tabela["DatCompetencia"].dt.year,
        mes=lambda tabela: tabela["DatCompetencia"].dt.month,
        nome_mes=lambda tabela: tabela["DatCompetencia"].dt.strftime("%B"),
        trimestre=lambda tabela: tabela["DatCompetencia"].dt.quarter,
        semestre=lambda tabela: ((tabela["DatCompetencia"].dt.month - 1) // 6) + 1,
    )
    [["tempo_id", "DatCompetencia", "ano", "mes", "nome_mes", "trimestre", "semestre"]]
    .rename(columns={"DatCompetencia": "data_competencia"})
)

# Dimensão distribuidora

dim_distribuidora = (
    silver_df[["NumCNPJAgenteDistribuidora", "SigAgenteDistribuidora", "NomAgenteDistribuidora"]]
    .drop_duplicates()
    .sort_values(["SigAgenteDistribuidora", "NomAgenteDistribuidora"])
    .reset_index(drop=True)
    .assign(distribuidora_id=lambda tabela: tabela.index + 1)
    [["distribuidora_id", "NumCNPJAgenteDistribuidora", "SigAgenteDistribuidora", "NomAgenteDistribuidora"]]
)

# Dimensão acessante

dim_acessante = (
    silver_df[["NumCNPJAgenteAcessante", "IdeAgenteAcessante", "NomAgenteAcessante"]]
    .drop_duplicates()
    .sort_values(["NomAgenteAcessante", "NumCNPJAgenteAcessante"])
    .reset_index(drop=True)
    .assign(acessante_id=lambda tabela: tabela.index + 1)
    [["acessante_id", "NumCNPJAgenteAcessante", "IdeAgenteAcessante", "NomAgenteAcessante"]]
)

# Dimensão mercado

dim_mercado = (
    silver_df[[
        "NomTipoMercado",
        "DscModalidadeTarifaria",
        "DscSubGrupoTarifario",
        "DscClasseConsumoMercado",
        "DscSubClasseConsumidor",
        "DscDetalheConsumidor",
        "DscPostoTarifario",
        "DscOpcaoEnergia",
        "DscDetalheMercado",
    ]]
    .drop_duplicates()
    .sort_values([
        "NomTipoMercado",
        "DscModalidadeTarifaria",
        "DscPostoTarifario",
        "DscDetalheMercado",
    ])
    .reset_index(drop=True)
    .assign(mercado_id=lambda tabela: tabela.index + 1)
    [[
        "mercado_id",
        "NomTipoMercado",
        "DscModalidadeTarifaria",
        "DscSubGrupoTarifario",
        "DscClasseConsumoMercado",
        "DscSubClasseConsumidor",
        "DscDetalheConsumidor",
        "DscPostoTarifario",
        "DscOpcaoEnergia",
        "DscDetalheMercado",
    ]]
)

# Tabela fato
fato_energia = silver_df[[
    "DatGeracaoConjuntoDados",
    "DatCompetencia",
    "NumCNPJAgenteDistribuidora",
    "SigAgenteDistribuidora",
    "NomAgenteDistribuidora",
    "NumCNPJAgenteAcessante",
    "IdeAgenteAcessante",
    "NomAgenteAcessante",
    "NomTipoMercado",
    "DscModalidadeTarifaria",
    "DscSubGrupoTarifario",
    "DscClasseConsumoMercado",
    "DscSubClasseConsumidor",
    "DscDetalheConsumidor",
    "DscPostoTarifario",
    "DscOpcaoEnergia",
    "DscDetalheMercado",
    "VlrMercado",
    "arquivo_origem",
]].copy()

fato_energia = fato_energia.merge(
    dim_tempo[["tempo_id", "data_competencia"]],
    left_on="DatCompetencia",
    right_on="data_competencia",
    how="left",
    validate="m:1",
)
fato_energia = fato_energia.merge(
    dim_distribuidora[["distribuidora_id", "NumCNPJAgenteDistribuidora", "SigAgenteDistribuidora", "NomAgenteDistribuidora"]],
    on=["NumCNPJAgenteDistribuidora", "SigAgenteDistribuidora", "NomAgenteDistribuidora"],
    how="left",
    validate="m:1",
)
fato_energia = fato_energia.merge(
    dim_acessante[["acessante_id", "NumCNPJAgenteAcessante", "IdeAgenteAcessante", "NomAgenteAcessante"]],
    on=["NumCNPJAgenteAcessante", "IdeAgenteAcessante", "NomAgenteAcessante"],
    how="left",
    validate="m:1",
)
fato_energia = fato_energia.merge(
    dim_mercado[[
        "mercado_id",
        "NomTipoMercado",
        "DscModalidadeTarifaria",
        "DscSubGrupoTarifario",
        "DscClasseConsumoMercado",
        "DscSubClasseConsumidor",
        "DscDetalheConsumidor",
        "DscPostoTarifario",
        "DscOpcaoEnergia",
        "DscDetalheMercado",
    ]],
    on=[
        "NomTipoMercado",
        "DscModalidadeTarifaria",
        "DscSubGrupoTarifario",
        "DscClasseConsumoMercado",
        "DscSubClasseConsumidor",
        "DscDetalheConsumidor",
        "DscPostoTarifario",
        "DscOpcaoEnergia",
        "DscDetalheMercado",
    ],
    how="left",
    validate="m:1",
)

fato_energia = fato_energia.drop(columns=["data_competencia"])
fato_energia = fato_energia.assign(fato_id=lambda tabela: tabela.index + 1)[[
    "fato_id",
    "tempo_id",
    "distribuidora_id",
    "acessante_id",
    "mercado_id",
    "DatGeracaoConjuntoDados",
    "VlrMercado",
    "arquivo_origem",
]]

# Audotoria de integridade das chaves primarias

print("Auditoria de integridade (verificar chaves unicas)")

dimensoes_checagem = {
    "Dim Distribuidora": (dim_distribuidora, "distribuidora_id"),
    "Dim Acessante": (dim_acessante, "acessante_id"),
    "Dim Mercado": (dim_mercado, "mercado_id"),
    "Dim Tempo": (dim_tempo, "tempo_id")
}

for nome_dim, (dataframe, coluna_chave) in dimensoes_checagem.items():
    total_linhas = len(dataframe)
    valores_unicos = dataframe[coluna_chave].nunique()
    possui_nulos = dataframe[coluna_chave].isnull().any()
    
    if total_linhas == valores_unicos and not possui_nulos:
        print(f"[OK] {nome_dim}: Chave '{coluna_chave}' é integra (Unica e sem nulos).")
    else:
        print(f"[ALERTA CRITICO] {nome_dim}: Falha de integridade! Linhas: {total_linhas} | Unicos: {valores_unicos} | Contem nulos: {possui_nulos}")

# Validacao de relacionamento fato x dimensao
print("\nVerificacao de consistencia da tabela fato")
chaves_fato = ["tempo_id", "distribuidora_id", "acessante_id", "mercado_id"]

for chave in chaves_fato:
    nulos_na_fato = fato_energia[chave].isnull().sum()
    if nulos_na_fato > 0:
        print(f"[ALERTA] A coluna '{chave}' possui {nulos_na_fato} registros sem correspondencia nas dimensoes!")
    else:
        print(f"[OK] Integração da chave '{chave}' realizada com sucesso.")
        

# Exporta Bronze, Silver e o esquema estrela com CSV separado por virgula.
df.to_csv(bronze_path, index=False, encoding="utf-8-sig")
silver_df.to_csv(silver_path, index=False, encoding="utf-8-sig")
dim_tempo.to_csv(dim_tempo_path, index=False, encoding="utf-8-sig")
dim_distribuidora.to_csv(dim_distribuidora_path, index=False, encoding="utf-8-sig")
dim_acessante.to_csv(dim_acessante_path, index=False, encoding="utf-8-sig")
dim_mercado.to_csv(dim_mercado_path, index=False, encoding="utf-8-sig")
fato_energia.to_csv(fato_energia_path, index=False, encoding="utf-8-sig")

{
    "bronze_path": str(bronze_path),
    "silver_path": str(silver_path),
    "dim_tempo_path": str(dim_tempo_path),
    "dim_distribuidora_path": str(dim_distribuidora_path),
    "dim_acessante_path": str(dim_acessante_path),
    "dim_mercado_path": str(dim_mercado_path),
    "fato_energia_path": str(fato_energia_path),
}

Auditoria de integridade (verificar chaves unicas)
[OK] Dim Distribuidora: Chave 'distribuidora_id' é integra (Unica e sem nulos).
[OK] Dim Acessante: Chave 'acessante_id' é integra (Unica e sem nulos).
[OK] Dim Mercado: Chave 'mercado_id' é integra (Unica e sem nulos).
[OK] Dim Tempo: Chave 'tempo_id' é integra (Unica e sem nulos).

Verificacao de consistencia da tabela fato
[OK] Integração da chave 'tempo_id' realizada com sucesso.
[OK] Integração da chave 'distribuidora_id' realizada com sucesso.
[OK] Integração da chave 'acessante_id' realizada com sucesso.
[OK] Integração da chave 'mercado_id' realizada com sucesso.


{'bronze_path': '../Datas/processed/bronze/bronze_dataset.csv',
 'silver_path': '../Datas/processed/silver/silver_dataset.csv',
 'dim_tempo_path': '../Datas/processed/gold/dim_tempo.csv',
 'dim_distribuidora_path': '../Datas/processed/gold/dim_distribuidora.csv',
 'dim_acessante_path': '../Datas/processed/gold/dim_acessante.csv',
 'dim_mercado_path': '../Datas/processed/gold/dim_mercado.csv',
 'fato_energia_path': '../Datas/processed/gold/fato_energia.csv'}

**5. Validação final do ETL**

Aqui conferimos o volume final das dimensões e da tabela fato geradas no esquema estrela.

In [16]:
print(f"Bronze linhas: {len(df):,}")
print(f"Silver linhas: {len(silver_df):,}")
print(f"Dim tempo linhas: {len(dim_tempo):,}")
print(f"Dim distribuidora linhas: {len(dim_distribuidora):,}")
print(f"Dim acessante linhas: {len(dim_acessante):,}")
print(f"Dim mercado linhas: {len(dim_mercado):,}")
print(f"Fato energia linhas: {len(fato_energia):,}")

display(dim_tempo.head())
display(dim_distribuidora.head())
display(dim_acessante.head())
display(dim_mercado.head())
display(fato_energia.head())

Bronze linhas: 3,029,329
Silver linhas: 3,029,186
Dim tempo linhas: 28
Dim distribuidora linhas: 103
Dim acessante linhas: 1,643
Dim mercado linhas: 12,902
Fato energia linhas: 3,029,186


,tempo_id,data_competencia,ano,mes,nome_mes,trimestre,semestre
0,1,2024-01-01,2024,1,January,1,1
1,2,2024-02-01,2024,2,February,1,1
2,3,2024-03-01,2024,3,March,1,1
3,4,2024-04-01,2024,4,April,2,1
4,5,2024-05-01,2024,5,May,2,1


,distribuidora_id,NumCNPJAgenteDistribuidora,SigAgenteDistribuidora,NomAgenteDistribuidora
0,1,02341467000120,AME,AMAZONAS ENERGIA S.A
1,2,02341470000144,BOA VISTA,RORAIMA ENERGIA S.A.
2,3,30460297000139,CASTRO-DIS,COOPERATIVA DE DISTRIBUICAO DE ENERGIA ELETRIC...
3,4,05965546000109,CEA,COMPANHIA DE ELETRICIDADE DO AMAPA CEA
4,5,60196987000193,CEDRAP,COOPERATIVA DE ELETRIFICACAO DA REGIAO DO ALTO...


,acessante_id,NumCNPJAgenteAcessante,IdeAgenteAcessante,NomAgenteAcessante
0,1,0,Nao Informado,ABAÚNA
1,2,0,Nao Informado,ABRANJO I
2,3,0,Nao Informado,ACOMAIS
3,4,0,Nao Informado,ADO POPINHAKI
4,5,0,Nao Informado,ADVOGADO EDUARDO SOARES (ANTIGA BOA ESPERANÇA)


,mercado_id,NomTipoMercado,DscModalidadeTarifaria,DscSubGrupoTarifario,DscClasseConsumoMercado,DscSubClasseConsumidor,DscDetalheConsumidor,DscPostoTarifario,DscOpcaoEnergia,DscDetalheMercado
0,1,REFATURAMENTO - REGULAR,AZUL,A4,SERVIÇO PÚBLICO,"ÁGUA, ESGOTO E SANEAMENTO",NÃO SE APLICA,FORA PONTA,CATIVO,DEMANDA FATURADA (KW)
1,2,REFATURAMENTO - REGULAR,AZUL,A3,INDUSTRIAL,NÃO SE APLICA,NÃO SE APLICA,FORA PONTA,LIVRE,DEMANDA FATURADA (KW)
2,3,REFATURAMENTO - REGULAR,AZUL,A3A,COMERCIAL,NÃO SE APLICA,NÃO SE APLICA,FORA PONTA,CATIVO,DEMANDA FATURADA (KW)
3,4,REFATURAMENTO - REGULAR,AZUL,A2,SERVIÇO PÚBLICO,TRAÇÃO ELÉTRICA,NÃO SE APLICA,FORA PONTA,CATIVO,DEMANDA FATURADA (KW)
4,5,REFATURAMENTO - REGULAR,AZUL,A4,PODER PÚBLICO,NÃO SE APLICA,NÃO SE APLICA,FORA PONTA,CATIVO,DEMANDA FATURADA (KW)


,fato_id,tempo_id,distribuidora_id,acessante_id,mercado_id,DatGeracaoConjuntoDados,VlrMercado,arquivo_origem
0,1,1,70,1009,12572,2026-05-15,583151.00,SAMP-2024.CSV
1,2,1,70,1009,3633,2026-05-15,5431.29,SAMP-2024.CSV
2,3,1,70,1009,3036,2026-05-15,28137.26,SAMP-2024.CSV
3,4,1,70,1009,8557,2026-05-15,235.94,SAMP-2024.CSV
4,5,1,70,1009,3686,2026-05-15,2061470.44,SAMP-2024.CSV


In [ ]:
#Instalando as dependências necessárias
!pip install sqlalchemy
!pip install psycopg2-binary

In [ ]:
from sqlalchemy import create_engine

# Preencher com Dados no Postgree

usuario_db = 'seu_usuario'           
senha_db = 'sua_senha'               
host_db = 'seu_host'                 
porta_db = 'sua_porta'               
nome_banco_db = 'seu_nome_do_banco' 

# Criando a string de conexao
DATABASE_URL = f'postgresql://{usuario_db}:{senha_db}@{host_db}:{porta_db}/{nome_banco_db}?client_encoding=utf8'

print("Tentando criar conexão com o banco PostgreSQL...")

try:
    # Estabelecendo conexão ativa
    engine = create_engine(DATABASE_URL)
    print("Conexao bem-sucedida! Iniciando a carga do Esquema Estrela no Postgres...\n")
    
    # Carregando as  Dimensoes no Postgres
    
    print("Carregando Dimensao Tempo...")
    dim_tempo.to_sql('dim_tempo', con=engine, if_exists='replace', index=False)
    
    print("Carregando Dimensao Distribuidora...")
    dim_distribuidora.to_sql('dim_distribuidora', con=engine, if_exists='replace', index=False)
    
    print("Carregando Dimensao Acessante...")
    dim_acessante.to_sql('dim_acessante', con=engine, if_exists='replace', index=False)
    
    print("Carregando Dimensao Mercado...")
    dim_mercado.to_sql('dim_mercado', con=engine, if_exists='replace', index=False)
    
    # Carregando Tabela Fato no Postgres
    
    print("Carregando Tabela Fato Energia...")
    fato_energia.to_sql('fato_energia', con=engine, if_exists='replace', index=False)

    print("\n CARGA CONCLUÍDA! Todas as tabelas do Esquema Estrela foram criadas no seu PostgreSQL.")

except Exception as e:
    print(f"\n--- [ERRO CRÍTICO] ---")
    print(f"Erro ao conectar ou carregar os dados no PostgreSQL: {e}")